In [3]:
"""
================================================================================
SCRIPT 2: BUILD AND DISPLAY TABLE 2 FROM PER-TILE CSVs

Linear step-by-step workflow:
1. Set pixel area (km²), age rules, input CSV folder, and output TABLE1 folder.
2. Read the validation shapefile to map each pristine tile (mask_artif == 0) to a region.
3. Loop through each pristine tile's CSV (no rasters are read here):
   - Total area in 2025 (km²): class == "gain" and 1 <= age <= 41
   - Landward expansion (km²): class == "gain", age < 41, position == "land"
   - Seaward expansion (km²): class == "gain", age < 41, position == "sea"
   - Landward loss (km²): class == "loss", position == "land"
   - Seaward loss (km²): class == "loss", position == "sea"
   - Mangrove cover 1985 (km²): Total 2025 - (Landward exp + Seaward exp)
     + (Landward loss + Seaward loss)
4. Group by region, sort by descending landward expansion, append a "Global" row,
   format "XX.X (YY.Y%)" shares (% of 1985 cover for gains; % of 2025 cover
   for losses), print Table 2, and save CSV files.
================================================================================
"""

from pathlib import Path
import warnings
import geopandas as gpd
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------------------
# 1. SETTINGS AND PATHS
# ------------------------------------------------------------------------------
PIXEL_AREA_KM2 = (29.869 ** 2) / 1e6
TOTAL_2025_MIN_AGE = 1
TOTAL_2025_MAX_AGE = 41  # 1 to 41 years inclusive for Total area in 2025
EXPANSION_MAX_AGE_EXCLUSIVE = 41  # Strictly less than 41 (1 to 40) for expansion

SHP_VALIDATION_FILE = Path(
    r"D:\MANGROVES\Research_Article_Clean\Created_Data\gmwv4_mask_anthropogenic_validation.shp"
)
INPUT_CSV_FOLDER = Path(
    r"D:\MANGROVES\Research_Article_Clean\Created_Data\CSV_Gain_Loss_Age"
)
TABLE1_OUTPUT_FOLDER = Path(
    r"D:\MANGROVES\Research_Article_Clean\Table2"
)
TABLE1_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

COL_COVER_1985 = "Mangrove cover 1985 (km²)"
COL_TOTAL_2025 = "Mangrove cover 2025 (km²)"
COL_LANDWARD_EXP = "Landward gains (km²)"
COL_SEAWARD_EXP = "Seaward gains (km²)"
COL_LANDWARD_LOSS = "Landward losses (km²)"
COL_SEAWARD_LOSS = "Seaward losses (km²)"

NUMERIC_COLUMNS = [
    COL_COVER_1985,
    COL_TOTAL_2025,
    COL_LANDWARD_EXP,
    COL_SEAWARD_EXP,
    COL_LANDWARD_LOSS,
    COL_SEAWARD_LOSS,
]

# ------------------------------------------------------------------------------
# 2. LOAD PRISTINE TILES AND THEIR REGIONS
# ------------------------------------------------------------------------------
shp_table = gpd.read_file(SHP_VALIDATION_FILE)
id_col = "clean_id" if "clean_id" in shp_table.columns else "tile_id"

if "region" not in shp_table.columns:
    raise KeyError(f"Expected 'region' column in {SHP_VALIDATION_FILE}, found: {list(shp_table.columns)}")

tiles_df = pd.DataFrame({
    "tile_id": shp_table[id_col].astype(str).str.strip(),
    "region": shp_table["region"].astype(str).str.strip(),
    "mask_artif": pd.to_numeric(shp_table["mask_artif"], errors="coerce"),
})
tiles_df = tiles_df[
    (tiles_df["mask_artif"] == 0)
    & (tiles_df["tile_id"] != "")
    & (tiles_df["tile_id"].str.lower() != "nan")
    & (tiles_df["region"] != "")
    & (tiles_df["region"].str.lower() != "nan")
].drop_duplicates(subset=["tile_id"]).reset_index(drop=True)

print(
    f"Aggregating {len(tiles_df)} pristine tiles across "
    f"{tiles_df['region'].nunique()} regions from {INPUT_CSV_FOLDER}..."
)

# ------------------------------------------------------------------------------
# 3. SUMMARIZE EACH TILE FROM ITS CSV
# ------------------------------------------------------------------------------
tile_records = []

for idx, row in enumerate(tiles_df.itertuples(index=False), start=1):
    tile_id = row.tile_id
    region = row.region
    csv_path = INPUT_CSV_FOLDER / f"{tile_id}.csv"

    if not csv_path.exists():
        print(f"⚠️ [{tile_id}] CSV not found in {INPUT_CSV_FOLDER}; skipping.")
        continue

    px_df = pd.read_csv(csv_path)

    if px_df.empty:
        tile_records.append({
            "tile_id": tile_id,
            "region": region,
            COL_COVER_1985: 0.0,
            COL_TOTAL_2025: 0.0,
            COL_LANDWARD_EXP: 0.0,
            COL_SEAWARD_EXP: 0.0,
            COL_LANDWARD_LOSS: 0.0,
            COL_SEAWARD_LOSS: 0.0,
        })
    else:
        ages = pd.to_numeric(px_df["age"], errors="coerce")
        is_gain = px_df["class"] == "gain"
        is_loss = px_df["class"] == "loss"
        is_land = px_df["position"] == "land"
        is_sea = px_df["position"] == "sea"

        total_2025_mask = is_gain & (ages >= TOTAL_2025_MIN_AGE) & (ages <= TOTAL_2025_MAX_AGE)
        expansion_mask = is_gain & (ages < EXPANSION_MAX_AGE_EXCLUSIVE)

        total_2025_km2 = int(total_2025_mask.sum()) * PIXEL_AREA_KM2
        landward_exp_km2 = int((expansion_mask & is_land).sum()) * PIXEL_AREA_KM2
        seaward_exp_km2 = int((expansion_mask & is_sea).sum()) * PIXEL_AREA_KM2
        landward_loss_km2 = int((is_loss & is_land).sum()) * PIXEL_AREA_KM2
        seaward_loss_km2 = int((is_loss & is_sea).sum()) * PIXEL_AREA_KM2

        cover_1985_km2 = (
            total_2025_km2
            - (landward_exp_km2 + seaward_exp_km2)
            + (landward_loss_km2 + seaward_loss_km2)
        )

        tile_records.append({
            "tile_id": tile_id,
            "region": region,
            COL_COVER_1985: cover_1985_km2,
            COL_TOTAL_2025: total_2025_km2,
            COL_LANDWARD_EXP: landward_exp_km2,
            COL_SEAWARD_EXP: seaward_exp_km2,
            COL_LANDWARD_LOSS: landward_loss_km2,
            COL_SEAWARD_LOSS: seaward_loss_km2,
        })

    if idx % 25 == 0 or idx == len(tiles_df):
        print(f"Progress: {idx}/{len(tiles_df)} tiles checked ({len(tile_records)} valid)")

if not tile_records:
    raise RuntimeError("No valid tile CSVs found. Run Script 1 first.")

per_tile_df = pd.DataFrame(tile_records)

# ------------------------------------------------------------------------------
# 4. AGGREGATE BY REGION, ADD GLOBAL ROW, AND FORMAT TABLE 2
# ------------------------------------------------------------------------------
by_region_df = (
    per_tile_df.groupby("region", as_index=True)[NUMERIC_COLUMNS]
    .sum()
    .sort_values(by=COL_LANDWARD_EXP, ascending=False)
)

global_df = pd.DataFrame(
    [by_region_df[NUMERIC_COLUMNS].sum()],
    index=["Global"],
)

numeric_table = pd.concat([by_region_df, global_df], axis=0)
numeric_table.index.name = "region"

formatted_table = pd.DataFrame(index=numeric_table.index)
formatted_table.index.name = "region"
formatted_table[COL_COVER_1985] = numeric_table[COL_COVER_1985].map(lambda v: f"{v:.1f}")
formatted_table[COL_TOTAL_2025] = numeric_table[COL_TOTAL_2025].map(lambda v: f"{v:.1f}")

# Gains (%): relative to Mangrove cover 1985 (km²)
for col in [COL_LANDWARD_EXP, COL_SEAWARD_EXP]:
    formatted_cells = []
    for val, base_1985 in zip(numeric_table[col], numeric_table[COL_COVER_1985]):
        pct = (val / base_1985 * 100.0) if base_1985 > 0 else 0.0
        formatted_cells.append(f"{val:.1f} ({pct:.1f}%)")
    formatted_table[col] = formatted_cells

# Losses (%): relative to Total area in 2025 (km²)
for col in [COL_LANDWARD_LOSS, COL_SEAWARD_LOSS]:
    formatted_cells = []
    for val, base_2025 in zip(numeric_table[col], numeric_table[COL_TOTAL_2025]):
        pct = (val / base_2025 * 100.0) if base_2025 > 0 else 0.0
        formatted_cells.append(f"{val:.1f} ({pct:.1f}%)")
    formatted_table[col] = formatted_cells

# ------------------------------------------------------------------------------
# 5. SAVE AND DISPLAY TABLE 2
# ------------------------------------------------------------------------------
formatted_csv = TABLE1_OUTPUT_FOLDER / "Table2_formated.csv"
numeric_csv = TABLE1_OUTPUT_FOLDER / "Table2.csv"

formatted_table.to_csv(formatted_csv)
numeric_table.round(1).to_csv(numeric_csv)

print("\n" + "=" * 140)
print("TABLE 2 — MANGROVE COVER (1985 & 2025), EXPANSION (% OF 1985) & LOSS (% OF 2025)")
print("=" * 140)
print(formatted_table.to_string())
print("=" * 140)
print(f"Saved formatted table to: {formatted_csv}")
print(f"Saved numeric table to:   {numeric_csv}")

Aggregating 1261 pristine tiles across 11 regions from D:\MANGROVES\Research_Article_Clean\Created_Data\CSV_Gain_Loss_Age...
Progress: 25/1261 tiles checked (25 valid)
Progress: 50/1261 tiles checked (50 valid)
Progress: 75/1261 tiles checked (75 valid)
Progress: 100/1261 tiles checked (100 valid)
Progress: 125/1261 tiles checked (125 valid)
Progress: 150/1261 tiles checked (150 valid)
Progress: 175/1261 tiles checked (175 valid)
Progress: 200/1261 tiles checked (200 valid)
Progress: 225/1261 tiles checked (225 valid)
Progress: 250/1261 tiles checked (250 valid)
Progress: 275/1261 tiles checked (275 valid)
Progress: 300/1261 tiles checked (300 valid)
Progress: 325/1261 tiles checked (325 valid)
Progress: 350/1261 tiles checked (350 valid)
Progress: 375/1261 tiles checked (375 valid)
Progress: 400/1261 tiles checked (400 valid)
Progress: 425/1261 tiles checked (425 valid)
Progress: 450/1261 tiles checked (450 valid)
Progress: 475/1261 tiles checked (475 valid)
Progress: 500/1261 tiles c